### Salary Predictor Model

#### 1. Connect Python to BigQuery

In [ ]:
# Authenticate user account
from google.colab import auth
auth.authenticate_user()
print("Successfully authenticated.")

Successfully authenticated.


In [ ]:
from google.cloud import bigquery
import pandas as pd

# Define project ID (should match the exact ID in BigQuery)
project_id = "hys-job-postings-analysis"
client = bigquery.Client(project = project_id)

# SQL query to pull the cleaned data from BigQuery
query = """
SELECT * FROM `ai_job_market.cleaned_jobs`
"""

df = client.query(query).to_dataframe()
df.head()

,standardized_title,location,state_group,salary_range,experience_level,remote_status,skills,programming_languages
0,ML/AI Engineer,"Austin, TX",Other US,$80000 - $96000,Entry,Hybrid,"XGBoost, Supervised Learning, NumPy, Regressio...","Julia, C++, Python"
1,Other Tech Role,"Austin, TX",Other US,$91000 - $109000,Entry,Hybrid,"XGBoost, OpenCV, OCR, Tokenization, Object Det...","SQL, Julia, Java, Python"
2,ML/AI Engineer,"Austin, TX",Other US,$81000 - $101000,Entry,Hybrid,"XGBoost, Supervised Learning, NumPy, Regressio...","Julia, R, Python"
3,ML/AI Engineer,"Austin, TX",Other US,$93000 - $120000,Entry,Hybrid,"Fine-tuning, RAG (Retrieval-Augmented Generati...","Go, Python"
4,ML/AI Engineer,"Austin, TX",Other US,$83000 - $97000,Entry,Hybrid,"Docker, Triton Inference Server, NLTK, CI/CD, ...","Python, SQL, Go"


#### 2. "Salary Cleansing" Function
When working in BigQuery and writing the SQL query to check the cleansed table I created, it returned California salary ranges with British pound signs instead of dollar signs $. This is because of the BigQuery’s automatic character encoding when I uploaded the CSV using Character Map V2.

Now I'm going to create a Python function to strip away those $ and £ symbols, clean up the commas, split the ranges, and turn them into single, clean average numbers.

In [ ]:
import re

def clean_salary_range(salary_str):
  if pd.isna(salary_str):
    return None

  # regex to strip out everything except numbers and dashes
  # removes $, £, commas, spaces, and text like "+ 1.0% Equity"
  cleaned = re.sub(r'[^0-9\-]', '', str(salary_str))

  # split the string by the dash to separate the low and high of the salary
  parts = cleaned.split('-')

  try:
    if len(parts) == 2:
      low = float(parts[0])
      high = float(parts[1])
      return (low + high) / 2.0   # average of the range
    elif len(parts) == 1 and parts[0] != '':
      return float(parts[0])      # if it's just a single number, return it
  except ValueError:
    return None
  return None

# apply cleaning function to create a brand new numeric 'salary' column
df['salary'] = df['salary_range'].apply(clean_salary_range)

# drop any rows where salary couldn't be parsed
df = df.dropna(subset = ['salary'])

# displaying new clean salary column next to old one to verify that it worked
df[['salary', 'salary_range']].head()


,salary,salary_range
0,88000.0,$80000 - $96000
1,100000.0,$91000 - $109000
2,91000.0,$81000 - $101000
3,106500.0,$93000 - $120000
4,90000.0,$83000 - $97000


#### 3. Training the ML Model
Data has now been fully cleaned (in BigQuery and here).
Some columns like standardized_title, experience_level, and state_group are text (categorical), but ML models can only read numbers.

=> will use One-Hot Encoding to turn them into numeric columns, split data into training and testing sets, and train a Random Forest Regressor.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# select feature variables (X) and target variable (y)
X = df[['standardized_title', 'experience_level', 'state_group', 'remote_status']]
y = df['salary']

# convert text columns into numeric columns using One-Hot Encoding
X_encoded = pd.get_dummies(X, drop_first = True)

# split 80% train data, 20% test data
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size = 0.2, random_state = 42)

# intialize and train the random forest model
model = RandomForestRegressor(n_estimators = 100, random_state = 42)
model.fit(X_train, y_train)

# make predictions on test data to evaluate it
predictions = model.predict(X_test)

# calculate accuracy
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Model Training Complete.")
print(f"Mean absolute error: ${mae:,.2f}")
print(f"R² score (variance explained): {r2:.2f}")

Model Training Complete.
Mean absolute error: $24,737.41
R² score (variance explained): 0.60


#### 4. Save model

Tried a few things out to try and get better model scores but an R2 score of 60 was the best I got.

In [ ]:
# Use encoded data matrix to generate the final predictions
df['predicted_salary'] = model.predict(X_encoded)
df['predicted_salary'] = df['predicted_salary'].round(0)

# keep the best text columns for Tableau visualizations
tableau_df = df[['standardized_title', 'location', 'state_group', 'experience_level', 'remote_status',
                 'salary', 'predicted_salary', 'programming_languages']]

# export to a clean csv
tableau_df.to_csv('final_salary_predictions.csv', index = False)
print('CSV file generated')

CSV file generated
